# Financial Operations Analytics - Data Ingestion and Quality Pipeline

This notebook loads the raw financial datasets from the repository, validates their structure and values, cleans them, and saves standardized outputs for downstream use.

## Scope

The workflow is limited to:
- Environment setup and library imports
- Project configuration
- Dataset loading
- Data inspection
- Data validation
- Data quality checks
- Datatype conversion
- Missing value handling
- Duplicate handling
- Saving cleaned datasets


In [1]:
import logging
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('financial_data_pipeline')

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Unable to locate project root from current working directory.')

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info('Project root: %s', PROJECT_ROOT)
logger.info('Raw data directory: %s', RAW_DATA_DIR)
logger.info('Output directory: %s', OUTPUT_DIR)
print('Environment setup complete.')


2026-07-08 15:54:11,066 - INFO - Project root: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics
2026-07-08 15:54:11,067 - INFO - Raw data directory: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/raw
2026-07-08 15:54:11,067 - INFO - Output directory: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed


Environment setup complete.


## Dataset Inventory

The raw data folder contains three CSV files:
- financial_customers.csv
- financial_transactions.csv
- monthly_revenue.csv


In [2]:
RAW_FILES = {
    'customers': RAW_DATA_DIR / 'financial_customers.csv',
    'transactions': RAW_DATA_DIR / 'financial_transactions.csv',
    'monthly_revenue': RAW_DATA_DIR / 'monthly_revenue.csv',
}

for name, path in RAW_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f'Required dataset not found: {path}')

logger.info('Raw data files located successfully.')
print('Raw data files located successfully.')


2026-07-08 15:54:11,083 - INFO - Raw data files located successfully.


Raw data files located successfully.


In [3]:
def load_dataset(file_path: Path) -> pd.DataFrame:
    """Load a CSV dataset with explicit error handling."""
    try:
        df = pd.read_csv(file_path)
        logger.info('Loaded %s with shape %s', file_path.name, df.shape)
        return df
    except Exception as exc:
        logger.exception('Failed to load %s', file_path.name)
        raise RuntimeError(f'Unable to load dataset {file_path.name}: {exc}') from exc

def inspect_dataset(df: pd.DataFrame, name: str) -> None:
    """Print a concise inspection summary for a dataset."""
    print(f'\n===== {name} =====')
    print(f'Rows: {df.shape[0]:,}')
    print(f'Columns: {df.shape[1]}')
    print('Columns:')
    for column in df.columns:
        print(f' - {column}')
    print('\nPreview:')
    print(df.head(3).to_string(index=False))
    print('\nData types:')
    print(df.dtypes.astype(str).to_string())
    print('\nMissing values:')
    print(df.isna().sum().to_string())

customers_df = load_dataset(RAW_FILES['customers'])
transactions_df = load_dataset(RAW_FILES['transactions'])
mrr_df = load_dataset(RAW_FILES['monthly_revenue'])

inspect_dataset(customers_df, 'financial_customers.csv')
inspect_dataset(transactions_df, 'financial_transactions.csv')
inspect_dataset(mrr_df, 'monthly_revenue.csv')


2026-07-08 15:54:11,159 - INFO - Loaded financial_customers.csv with shape (20000, 36)
2026-07-08 15:54:12,041 - INFO - Loaded financial_transactions.csv with shape (329202, 30)
2026-07-08 15:54:12,050 - INFO - Loaded monthly_revenue.csv with shape (36, 18)



===== financial_customers.csv =====
Rows: 20,000
Columns: 36
Columns:
 - customer_id
 - signup_date
 - customer_age
 - country
 - region
 - city
 - industry
 - company_size
 - subscription_plan
 - contract_type
 - contract_length_months
 - monthly_recurring_revenue
 - payment_method
 - support_tickets
 - usage_score
 - login_frequency
 - nps_score
 - discount_percentage
 - tenure_months
 - customer_segment
 - customer_lifetime_value
 - customer_acquisition_cost
 - acquisition_channel
 - acquisition_month
 - is_active
 - churn
 - last_transaction_date
 - days_since_last_transaction
 - frequency_score
 - monetary_score
 - average_transaction_value
 - total_transactions
 - total_revenue
 - total_profit
 - customer_profitability
 - risk_score

Preview:
customer_id signup_date  customer_age country region             city   industry company_size subscription_plan contract_type  contract_length_months  monthly_recurring_revenue payment_method  support_tickets  usage_score  login_frequency  

## Validation Rules

The following checks are applied before cleaning:
- Required columns must be present
- Date-like columns must be parseable
- Numeric columns must be convertible to numeric values
- A business identifier should be unique
- Missing values must be tracked and handled consistently


In [4]:
def validate_dataset(df: pd.DataFrame, name: str) -> None:
    """Validate core quality expectations for a dataset."""
    issues = []

    if df.empty:
        issues.append('Dataset is empty')

    if name == 'customers':
        required_columns = [
            'customer_id', 'signup_date', 'country', 'industry', 'subscription_plan',
            'monthly_recurring_revenue', 'is_active', 'churn'
        ]
    elif name == 'transactions':
        required_columns = [
            'transaction_id', 'customer_id', 'transaction_date', 'gross_amount',
            'net_revenue', 'transaction_status'
        ]
    else:
        required_columns = [
            'year_month', 'gross_revenue', 'net_revenue', 'active_customers',
            'new_customers', 'churned_customers'
        ]

    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        issues.append(f'Missing required columns: {missing_columns}')

    if name == 'customers' and 'customer_id' in df.columns:
        duplicates = df['customer_id'].duplicated().sum()
        if duplicates:
            issues.append(f'Customer ID duplicates detected: {duplicates}')
    elif name == 'transactions' and 'transaction_id' in df.columns:
        duplicates = df['transaction_id'].duplicated().sum()
        if duplicates:
            issues.append(f'Transaction ID duplicates detected: {duplicates}')
    elif name == 'monthly_revenue' and 'year_month' in df.columns:
        duplicates = df['year_month'].duplicated().sum()
        if duplicates:
            issues.append(f'Year-month duplicates detected: {duplicates}')

    if issues:
        raise ValueError(f'Validation failed for {name}: {issues}')

    for column in ['signup_date', 'transaction_date', 'year_month']:
        if column in df.columns:
            pd.to_datetime(df[column], errors='raise')

    logger.info('Validation passed for %s', name)
    print(f'Validation passed for {name}.')

validate_dataset(customers_df, 'customers')
validate_dataset(transactions_df, 'transactions')
validate_dataset(mrr_df, 'monthly_revenue')


2026-07-08 15:54:12,297 - INFO - Validation passed for customers
2026-07-08 15:54:12,361 - INFO - Validation passed for transactions
2026-07-08 15:54:12,362 - INFO - Validation passed for monthly_revenue


Validation passed for customers.
Validation passed for transactions.
Validation passed for monthly_revenue.


## Data Type Conversion

Date columns are parsed to datetime and numeric columns are standardized for reliable downstream processing.


In [5]:
def convert_data_types(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Convert columns to appropriate dtypes."""
    converted = df.copy()

    if name == 'customers':
        for column in ['signup_date', 'last_transaction_date']:
            if column in converted.columns:
                converted[column] = pd.to_datetime(converted[column], errors='coerce')

        numeric_columns = [
            'customer_age', 'contract_length_months', 'monthly_recurring_revenue',
            'support_tickets', 'usage_score', 'login_frequency', 'nps_score',
            'discount_percentage', 'tenure_months', 'customer_lifetime_value',
            'customer_acquisition_cost', 'days_since_last_transaction', 'frequency_score',
            'monetary_score', 'average_transaction_value', 'total_transactions',
            'total_revenue', 'total_profit', 'risk_score'
        ]
    elif name == 'transactions':
        converted['transaction_date'] = pd.to_datetime(converted['transaction_date'], errors='coerce')
        numeric_columns = [
            'gross_amount', 'discount', 'tax', 'processing_fee', 'refund_amount',
            'support_cost', 'marketing_cost', 'operational_cost', 'net_revenue',
            'profit', 'profit_margin', 'payment_delay_days'
        ]
    else:
        converted['year_month'] = pd.to_datetime(converted['year_month'], errors='coerce')
        numeric_columns = [
            'gross_revenue', 'net_revenue', 'profit', 'profit_margin',
            'active_customers', 'new_customers', 'churned_customers',
            'transaction_count', 'average_order_value', 'growth_rate',
            'forecast_target'
        ]

    for column in numeric_columns:
        if column in converted.columns:
            converted[column] = pd.to_numeric(converted[column], errors='coerce')

    for column in converted.columns:
        if converted[column].dtype == 'object':
            converted[column] = converted[column].astype('string')

    logger.info('Converted dtypes for %s', name)
    return converted

customers_clean = convert_data_types(customers_df, 'customers')
transactions_clean = convert_data_types(transactions_df, 'transactions')
mrr_clean = convert_data_types(mrr_df, 'monthly_revenue')


2026-07-08 15:54:12,390 - INFO - Converted dtypes for customers
2026-07-08 15:54:12,534 - INFO - Converted dtypes for transactions
2026-07-08 15:54:12,536 - INFO - Converted dtypes for monthly_revenue


## Missing Value Handling

Missing values are filled with business-appropriate defaults where the meaning is clear and retained as missing where a replacement would be misleading.


In [6]:
def handle_missing_values(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Handle missing values in a dataset."""
    cleaned = df.copy()

    if name == 'customers':
        numeric_fill = {
            'support_tickets': 0.0,
            'usage_score': cleaned['usage_score'].median(),
            'nps_score': cleaned['nps_score'].median(),
        }
        for column, fill_value in numeric_fill.items():
            if column in cleaned.columns:
                cleaned[column] = cleaned[column].fillna(fill_value)
    elif name == 'transactions':
        if 'payment_delay_days' in cleaned.columns:
            cleaned['payment_delay_days'] = cleaned['payment_delay_days'].fillna(0.0)

    for column in cleaned.select_dtypes(include=['string']).columns:
        cleaned[column] = cleaned[column].fillna(pd.NA)

    logger.info('Missing values handled for %s', name)
    return cleaned

customers_clean = handle_missing_values(customers_clean, 'customers')
transactions_clean = handle_missing_values(transactions_clean, 'transactions')
mrr_clean = handle_missing_values(mrr_clean, 'monthly_revenue')


2026-07-08 15:54:12,559 - INFO - Missing values handled for customers
2026-07-08 15:54:12,747 - INFO - Missing values handled for transactions
2026-07-08 15:54:12,757 - INFO - Missing values handled for monthly_revenue


## Duplicate Handling

Duplicate rows are removed using the business identifier for each dataset, preserving the first occurrence.


In [7]:
def remove_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Remove duplicate rows based on dataset-specific identifiers."""
    cleaned = df.copy()

    if name == 'customers' and 'customer_id' in cleaned.columns:
        subset = ['customer_id']
    elif name == 'transactions' and 'transaction_id' in cleaned.columns:
        subset = ['transaction_id']
    elif name == 'monthly_revenue' and 'year_month' in cleaned.columns:
        subset = ['year_month']
    else:
        subset = None

    if subset is not None:
        duplicates_before = int(cleaned.duplicated(subset=subset).sum())
        cleaned = cleaned.drop_duplicates(subset=subset)
        logger.info('Removed %s duplicates from %s', duplicates_before, name)
        print(f'Removed {duplicates_before} duplicate rows from {name}.')
    else:
        logger.info('No duplicate key available for %s', name)

    return cleaned

customers_clean = remove_duplicates(customers_clean, 'customers')
transactions_clean = remove_duplicates(transactions_clean, 'transactions')
mrr_clean = remove_duplicates(mrr_clean, 'monthly_revenue')


2026-07-08 15:54:12,770 - INFO - Removed 0 duplicates from customers
2026-07-08 15:54:12,869 - INFO - Removed 0 duplicates from transactions
2026-07-08 15:54:12,879 - INFO - Removed 0 duplicates from monthly_revenue


Removed 0 duplicate rows from customers.
Removed 0 duplicate rows from transactions.
Removed 0 duplicate rows from monthly_revenue.


## Output Preparation and Export

The cleaned datasets are saved to the processed data folder for downstream workflows.


In [8]:
def save_cleaned_dataset(df: pd.DataFrame, file_name: str) -> None:
    """Persist a cleaned dataset to disk."""
    output_path = OUTPUT_DIR / file_name
    df.to_csv(output_path, index=False)
    logger.info('Saved cleaned dataset to %s', output_path)
    print(f'Saved: {output_path}')

save_cleaned_dataset(customers_clean, 'financial_customers_clean.csv')
save_cleaned_dataset(transactions_clean, 'financial_transactions_clean.csv')
save_cleaned_dataset(mrr_clean, 'monthly_revenue_clean.csv')

print('\nPipeline completed successfully.')
print('Cleaned datasets exported to:')
for path in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f' - {path.name}')


2026-07-08 15:54:13,041 - INFO - Saved cleaned dataset to /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/financial_customers_clean.csv


Saved: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/financial_customers_clean.csv


2026-07-08 15:54:15,057 - INFO - Saved cleaned dataset to /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/financial_transactions_clean.csv
2026-07-08 15:54:15,059 - INFO - Saved cleaned dataset to /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/monthly_revenue_clean.csv


Saved: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/financial_transactions_clean.csv
Saved: /Users/ayushkumarsingh/Downloads/Financial-Operations-Analytics/data/processed/monthly_revenue_clean.csv

Pipeline completed successfully.
Cleaned datasets exported to:
 - financial_customers_clean.csv
 - financial_transactions_clean.csv
 - monthly_revenue_clean.csv
